# CT-ScanEval: run the models in Google Colab

1. Add your API keys as Colab **Secrets** (the key icon in the left sidebar), named `ANTHROPIC_API_KEY`, `OPENAI_API_KEY`, `GEMINI_API_KEY`. Only add the ones you have, and turn on notebook access for each.
2. Run the cells top to bottom. The CPU runtime is enough. Nothing needs to be uploaded: the first cell copies the project from GitHub.

In [ ]:
!git clone https://github.com/Moyo8/-CT-ScanEval /content/CT-ScanEval
%cd /content/CT-ScanEval

In [ ]:
!pip install -q -r requirements.txt
!python -m pytest -q tests   # grader checks: should say 60 passed

In [ ]:
import os
from google.colab import userdata
for k in ['ANTHROPIC_API_KEY', 'OPENAI_API_KEY', 'GEMINI_API_KEY']:
    try:
        os.environ[k] = userdata.get(k)
        print(k, 'loaded')
    except Exception:
        print(k, 'not set (skip that provider)')

## Run the models
Each run is 8 tasks x 3 trials = 24 API calls. Model IDs change often; use the exact ID shown in each provider's model list and keep it the same for every run you report.

In [ ]:
# Claude (Anthropic)
!python run_eval.py --provider anthropic --model claude-opus-5-5 --effort high --trials 3

In [ ]:
# Optional: a second Claude model as a smaller/cheaper comparison
# !python run_eval.py --provider anthropic --model claude-haiku-5-5 --trials 3

In [ ]:
# Optional: OpenAI (replace with a current vision model ID from your account)
# !python run_eval.py --provider openai --model YOUR_OPENAI_MODEL_ID --trials 3

In [ ]:
# Optional: Gemini (replace with a current model ID from Google AI Studio)
# !python run_eval.py --provider gemini --model YOUR_GEMINI_MODEL_ID --trials 3

## Baselines and summary

In [ ]:
!python run_eval.py --provider random --trials 3
!python analysis/summarize.py --latest
from IPython.display import Image, display
display(Image('results/charts/per_task_scores.png'))

## Save results to GitHub
Colab deletes its files when the session ends, so this pushes `results/` and `transcripts/` back to the repository.

One-time setup: create a GitHub token (GitHub → Settings → Developer settings → Personal access tokens → **Fine-grained tokens** → Generate). Give it access to **only** the `-CT-ScanEval` repository with **Contents: Read and write**, then add it as a Colab Secret named `GITHUB_TOKEN`.

In [ ]:
import base64, subprocess
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
def git(*args):
    r = subprocess.run(['git', '-c', f'http.extraHeader=Authorization: Basic {auth}', *args], capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(token, '***').replace(auth, '***').strip())
    return r.returncode
git('config', 'user.name', 'Moyo8')
git('config', 'user.email', 'Moyo8@users.noreply.github.com')
git('add', 'results', 'transcripts')
git('commit', '-m', 'Model runs from Colab')
git('pull', '--rebase', 'origin', 'main')
print('pushed OK' if git('push', 'origin', 'HEAD:main') == 0 else 'PUSH FAILED - check the token')